### config

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import config.ConnectionConfig as cc
cc.setupEnvironment()

spark = cc.startLocalCluster("fact_rides",7)
spark.getActiveSession()

bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
25/03/19 13:56:09 WARN Utils: Your hostname, system resolves to a loopback address: 127.0.1.1; using 10.140.67.105 instead (on interface wlp170s0)
25/03/19 13:56:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/souls/.ivy2/cache
The jars for the packages stored in: /home/souls/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
org.elasticsearch#elasticsearch-spark-30_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f07e70c2-f6f3-4d79-b8c7-444987ca3901;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/souls/Projects/data4/spark-3.5.4-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
	found org.postgresql#postgresql;42.7.4 in central
	found org.checkerframework#checker-qual;3.42.0 in central
	found org.elasticsearch#elasticsearch-spark-30_2.12;8.15.2 in central
	found org.scala-lang#scala-reflect;2.

# EXTRACT

I tried using the spark API first, but there are some limitations on their joining of columns.

In [23]:
# EXTRACT rides:
# df_rides = spark.read.format("jdbc")\
#     .option("driver" , cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", "(SELECT * FROM rides) as rides_table") \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "rideid") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()
#
# #df_rides.printSchema()
# #df_rides.show()
# df_rides.count()

# EXTRACT bike_type through bike_lot through vehicle:
# vehicle_table_SQL = '(SELECT * FROM vehicles v LEFT JOIN bikelots l ON v.bikelotid = l.bikelotid) as vehicle_table'
# df_vehicles = spark.read.format("jdbc")\
#     .option("driver" , cc.get_Property("driver")) \
#     .option("url", cc.create_jdbc()) \
#     .option("dbtable", vehicle_table_SQL) \
#     .option("user", cc.get_Property("username")) \
#     .option("password", cc.get_Property("password")) \
#     .option("partitionColumn", "vehicleid") \
#     .option("numPartitions", 4) \
#     .option("lowerBound", 0)\
#     .option("upperBound", 100) \
#     .load()
#
# df_vehicles.show()

Py4JJavaError: An error occurred while calling o631.load.
: org.postgresql.util.PSQLException: ERROR: column "differentbikelotid" does not exist
  Position: 95
	at org.postgresql.core.v3.QueryExecutorImpl.receiveErrorResponse(QueryExecutorImpl.java:2733)
	at org.postgresql.core.v3.QueryExecutorImpl.processResults(QueryExecutorImpl.java:2420)
	at org.postgresql.core.v3.QueryExecutorImpl.execute(QueryExecutorImpl.java:372)
	at org.postgresql.jdbc.PgStatement.executeInternal(PgStatement.java:517)
	at org.postgresql.jdbc.PgStatement.execute(PgStatement.java:434)
	at org.postgresql.jdbc.PgPreparedStatement.executeWithFlags(PgPreparedStatement.java:194)
	at org.postgresql.jdbc.PgPreparedStatement.executeQuery(PgPreparedStatement.java:137)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.getQueryOutputSchema(JDBCRDD.scala:68)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.resolveTable(JDBCRDD.scala:58)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRelation$

Trying to do it in one SQL query:

In [ ]:
# EXTRACTING ALL IN ONE GO:
SQLstring = 'for real'

# TRANSFORM

# LOAD

In [ ]:
spark.stop()

# notes I didn't want to delete below:

In [ ]:
from pyspark.sql.functions import *
# https://spark.apache.org/docs/3.5.4/sql-data-sources-jdbc.html

df_rides = spark.read.format("jdbc")\
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "(SELECT * FROM rides) as rides_table") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0)\
    .option("upperBound", 100) \
    .load()
# limiting doesn't work, so i select 4,2 million rows...
df_rides.createOrReplaceTempView('listOfRides')
spark.sql("select * from listOfRides").show()

# lol

from pyspark.sql.functions import *
# https://spark.apache.org/docs/3.5.4/sql-data-sources-jdbc.html

df_stations = spark.read.format("jdbc")\
    .option("driver" , cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "(SELECT DISTINCT stationid, zipcode, district FROM stations) as stations_table") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "stationid") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0)\
    .option("upperBound", 100) \
    .load()
# limiting doesn't work, so i select 4,2 million rows...
df_stations.createOrReplaceTempView('listOfZipcodes')
spark.sql("select * from listOfZipcodes").show()

# more notes

# need to get rides and stations, for zip codes. JOIN those two.
# https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameReader.jdbc.html?highlight=option%20partitioncolumn